In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\DELL\Downloads\spam1.csv", encoding ="latin1")

In [3]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [4]:
df.shape

(6776, 5)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6776 entries, 0 to 6775
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          6776 non-null   object
 1   v2          6776 non-null   object
 2   Unnamed: 2  56 non-null     object
 3   Unnamed: 3  16 non-null     object
 4   Unnamed: 4  8 non-null      object
dtypes: object(5)
memory usage: 264.8+ KB


In [6]:
df= df.iloc[:,[0,1]]   #selecting the first 2 /removing unwanted cols

In [7]:
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
df = df.rename(columns={'v1':'Y' , 'v2':"X"})    # renaming columns

In [9]:
df.Y = df.Y.replace({'ham':0 , 'spam':1})    #replace ham and spam to 0 and 1

C:\Users\DELL\AppData\Local\Temp\ipykernel_16796\1335751850.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.Y = df.Y.replace({'ham':0 , 'spam':1})    #replace ham and spam to 0 and 1


In [10]:
df.Y.value_counts()

Y
0    5854
1     922
Name: count, dtype: int64

In [11]:
import nltk #natural language tool kit

In [12]:
from nltk.corpus import stopwords      #import stopwords

In [13]:
len(stopwords.words('english'))   #checking stopwords(words that do not add value)

198

In [14]:
import string                 #checking punctuation
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [15]:
df['X']=df['X'].str.lower()   #python is case sensitive as coverting everthing into one case 

In [16]:
df['X']

0       go until jurong point, crazy.. available only ...
1                           ok lar... joking wif u oni...
2       free entry in 2 a wkly comp to win fa cup fina...
3       u dun say so early hor... u c already then say...
4       nah i don't think he goes to usf, he lives aro...
                              ...                        
6771    this is the 2nd time we have tried 2 contact u...
6772                will ì_ b going to esplanade fr home?
6773    pity, * was in mood for that. so...any other s...
6774    the guy did some bitching but i acted like i'd...
6775                           rofl. its true to its name
Name: X, Length: 6776, dtype: object

In [17]:
def text_process(mess):          #creating a function
    """                           
    remove the punctuation
    remove the stopwords
    return the list of clean textwords
    """

    nopunc = [char for char in mess if char not in string.punctuation]          #punc
    nopunc ="".join(nopunc)

    return[word for word in nopunc.split() if word not in stopwords.words('english')]         #stopwords
              
    

In [18]:
df['X'].apply(text_process)     #applying the func to X variable to remove punc and stopwords

0       [go, jurong, point, crazy, available, bugis, n...
1                          [ok, lar, joking, wif, u, oni]
2       [free, entry, 2, wkly, comp, win, fa, cup, fin...
3           [u, dun, say, early, hor, u, c, already, say]
4       [nah, dont, think, goes, usf, lives, around, t...
                              ...                        
6771    [2nd, time, tried, 2, contact, u, u, å£750, po...
6772                   [ì, b, going, esplanade, fr, home]
6773                     [pity, mood, soany, suggestions]
6774    [guy, bitching, acted, like, id, interested, b...
6775                                   [rofl, true, name]
Name: X, Length: 6776, dtype: object

In [19]:
from sklearn.feature_extraction.text import CountVectorizer     #to convert word to num
#assign a ID / token to each word

In [20]:
bow_transformer = CountVectorizer(analyzer = text_process).fit(df['X']) 

In [22]:
len(bow_transformer.vocabulary_)     #after creating tdm we will have 9422 columns

9422

In [21]:
tdm = bow_transformer.transform(df['X'])

In [30]:
from sklearn.model_selection import train_test_split
train_x, test_x, train_y, test_y = train_test_split(tdm , df.Y)
#     tdm,
#     df["Y"],
#     test_size=0.20,
#     random_state=42,
#     stratify=df["Y"]
# ) 

In [31]:
from sklearn.metrics import confusion_matrix , classification_report

In [32]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(class_weight="balanced")
model.fit(train_x , train_y)
pred = model.predict(test_x)
print(confusion_matrix(test_y , pred))
print(classification_report(test_y , pred)) 

[[1465    2]
 [  22  205]]
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1467
           1       0.99      0.90      0.94       227

    accuracy                           0.99      1694
   macro avg       0.99      0.95      0.97      1694
weighted avg       0.99      0.99      0.99      1694



In [61]:
import pickle

# Save the EXACT trained model
with open(r"C:\Users\DELL\spam\spam_model.pkl", "wb") as file:
    pickle.dump(model, file)

# Save the EXACT fitted bow_transformer
with open(r"C:\Users\DELL\spam\vectorizer.pkl", "wb") as file:
    pickle.dump(bow_transformer, file)

In [63]:
with open(r"C:\Users\DELL\spam\spam_model.pkl", "rb") as file:
    test_model = pickle.load(file)

with open(r"C:\Users\DELL\spam\vectorizer.pkl", "rb") as file:
    test_vectorizer = pickle.load(file)

test_message = "Congratulations! You have won a free lottery prize. Claim your cash reward now!"

test_vector = test_vectorizer.transform([test_message])

print("Saved model prediction:", test_model.predict(test_vector))

Saved model prediction: [1]


In [34]:
test_message ="Congratulation! you won a free lottery!claim your cash reward now"
    

test_vector = bow_transformer.transform([test_message])

print("Jupyter prediction:", model.predict(test_vector))

Jupyter prediction: [0]


In [34]:
print(df['Y'].unique())
print(df['Y'].value_counts())

[0 1]
Y
0    5854
1     922
Name: count, dtype: int64


In [25]:

import pickle
import numpy as np

model_path = r"C:\Users\DELL\spam\spam_model.pkl"
vectorizer_path = r"C:\Users\DELL\spam\vectorizer.pkl"

with open(model_path, "rb") as file:
    model = pickle.load(file)

with open(vectorizer_path, "rb") as file:
    bow_transformer = pickle.load(file)

# Test message
test_message = """
Congratulations! You have won a free lottery prize.
Claim your cash reward now!
"""

# Convert text using your fitted vectorizer
test_vector = bow_transformer.transform([test_message])

# Prediction
prediction = model.predict(test_vector)[0]

# Probability for each class
probabilities = model.predict_proba(test_vector)[0]

# Find class indexes safely
classes = list(model.classes_)
spam_index = classes.index(1)
ham_index = classes.index(0)

spam_probability = probabilities[spam_index] * 100
ham_probability = probabilities[ham_index] * 100

print("=" * 45)
print("        SPAMSENTRY AI - PREDICTION")
print("=" * 45)

print("Predicted class:", "SPAM" if prediction == 1 else "HAM")
print(f"Spam probability: {spam_probability:.2f}%")
print(f"Ham probability:  {ham_probability:.2f}%")
print(f"Prediction confidence: {max(probabilities) * 100:.2f}%")
print("=" * 45)


        SPAMSENTRY AI - PREDICTION
Predicted class: SPAM
Spam probability: 76.85%
Ham probability:  23.15%
Prediction confidence: 76.85%
